<a href="https://colab.research.google.com/github/mesata/Facial-Expression-Recognition/blob/main/final_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import time
import wandb
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" აქტიური მოწყობილობა: {device}")

wandb.login()

 აქტიური მოწყობილობა: cuda


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mesatia (mesatia-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:

from google.colab import drive
drive.mount('/content/drive')

X_train_np = np.load('/content/drive/MyDrive/fer2013_processed/X_train.npy')
y_train_np = np.load('/content/drive/MyDrive/fer2013_processed/y_train.npy')
X_val_np = np.load('/content/drive/MyDrive/fer2013_processed/X_val.npy')
y_val_np = np.load('/content/drive/MyDrive/fer2013_processed/y_val.npy')
X_test_np = np.load('/content/drive/MyDrive/fer2013_processed/X_test.npy')
y_test_np = np.load('/content/drive/MyDrive/fer2013_processed/y_test.npy')

X_train_np = X_train_np.reshape(-1, 1, 48, 48) / 255.0
X_val_np = X_val_np.reshape(-1, 1, 48, 48) / 255.0
X_test_np = X_test_np.reshape(-1, 1, 48, 48) / 255.0

X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
X_val = torch.tensor(X_val_np, dtype=torch.float32)
y_val = torch.tensor(y_val_np, dtype=torch.long)
X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.long)

class_counts = np.bincount(y_train_np)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * 7
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f"მონაცემები მზადაა: Train size: {X_train.shape}, Test size: {X_test.shape}")

Mounted at /content/drive
მონაცემები მზადაა: Train size: torch.Size([24402, 1, 48, 48]), Test size: torch.Size([3589, 1, 48, 48])


In [6]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual
        return self.relu(out)

class ResNetFER(nn.Module):
    def __init__(self):
        super().__init__()
        self.init_conv = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.layer1 = ResidualBlock(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout2d(0.3)

        self.transit = nn.Conv2d(64, 128, kernel_size=1)

        self.layer2 = ResidualBlock(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.dropout2 = nn.Dropout2d(0.4)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 7)
        )

    def forward(self, x):
        x = self.init_conv(x)
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.dropout1(x)

        x = self.transit(x)
        x = self.layer2(x)
        x = self.pool2(x)
        x = self.dropout2(x)

        return self.classifier(x)

In [7]:
BATCH_SIZE = 128
LR = 0.001
EPOCHS = 15
WEIGHT_DECAY = 1e-4

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

temp_model = ResNetFER().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

temp_model.train()
start_f = time.time()
outputs = temp_model(images)
loss = criterion(outputs, labels)
f_time = time.time() - start_f

start_b = time.time()
loss.backward()
b_time = time.time() - start_b

print(f" time Forward: {f_time:.5f}s | Backward: {b_time:.5f}s")

 time Forward: 1.33730s | Backward: 0.23604s


In [8]:
run = wandb.init(
    project="facial-expression-recognition",
    name="ResNet_Invention_Run",
    config={
        "architecture": "ResNet-FER",
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "epochs": EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "forward_pass_time_sec": f_time,
        "backward_pass_time_sec": b_time
    }
)

model = ResNetFER().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(" training")
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        _, preds = outs.max(1)
        correct_train += preds.eq(lbls).sum().item()
        total_train += lbls.size(0)

    train_loss /= total_train
    train_acc = (correct_train / total_train) * 100

    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model(imgs)
            loss = criterion(outs, lbls)
            val_loss += loss.item() * imgs.size(0)
            _, preds = outs.max(1)
            correct_val += preds.eq(lbls).sum().item()
            total_val += lbls.size(0)

    val_loss /= total_val
    val_acc = (correct_val / total_val) * 100

    print(f"Epoch [{epoch}/{EPOCHS}] -> Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

torch.save(model.state_dict(), 'resnet_fer_best.pth')
run.finish()

 training
Epoch [1/15] -> Train Acc: 23.84% | Val Acc: 11.05%
Epoch [2/15] -> Train Acc: 35.57% | Val Acc: 16.11%
Epoch [3/15] -> Train Acc: 41.46% | Val Acc: 14.28%
Epoch [4/15] -> Train Acc: 44.55% | Val Acc: 16.62%
Epoch [5/15] -> Train Acc: 47.04% | Val Acc: 16.14%
Epoch [6/15] -> Train Acc: 49.04% | Val Acc: 14.42%
Epoch [7/15] -> Train Acc: 52.07% | Val Acc: 15.93%
Epoch [8/15] -> Train Acc: 54.16% | Val Acc: 14.00%
Epoch [9/15] -> Train Acc: 56.04% | Val Acc: 14.42%
Epoch [10/15] -> Train Acc: 57.87% | Val Acc: 14.28%
Epoch [11/15] -> Train Acc: 60.54% | Val Acc: 17.85%
Epoch [12/15] -> Train Acc: 63.06% | Val Acc: 17.09%
Epoch [13/15] -> Train Acc: 64.59% | Val Acc: 13.91%
Epoch [14/15] -> Train Acc: 66.95% | Val Acc: 17.32%
Epoch [15/15] -> Train Acc: 68.77% | Val Acc: 14.91%


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_accuracy,▁▃▄▄▅▅▅▆▆▆▇▇▇██
train_loss,█▇▆▅▅▄▄▃▃▃▂▂▂▁▁
val_accuracy,▁▆▄▇▆▄▆▄▄▄█▇▄▇▅
val_loss,█▁▂▁▁▁▃▅▂▅▂▁▂▂▂
epoch,15
train_accuracy,68.77305
train_loss,0.75719
val_accuracy,14.90597
val_loss,2.77182


ashkarad raghac dzaan avurie datashi, 68% trainze da 14 validationze ranairad movaxerxe aba. redoo

In [9]:
X_train_np = np.load('/content/drive/MyDrive/fer2013_processed/X_train.npy')
y_train_np = np.load('/content/drive/MyDrive/fer2013_processed/y_train.npy')
X_val_np = np.load('/content/drive/MyDrive/fer2013_processed/X_val.npy')
y_val_np = np.load('/content/drive/MyDrive/fer2013_processed/y_val.npy')
X_test_np = np.load('/content/drive/MyDrive/fer2013_processed/X_test.npy')
y_test_np = np.load('/content/drive/MyDrive/fer2013_processed/y_test.npy')

X_train_np = X_train_np.reshape(-1, 1, 48, 48)
X_val_np = X_val_np.reshape(-1, 1, 48, 48)
X_test_np = X_test_np.reshape(-1, 1, 48, 48)

if X_train_np.max() > 1.0:
    X_train_np = X_train_np / 255.0
if X_val_np.max() > 1.0:
    X_val_np = X_val_np / 255.0
if X_test_np.max() > 1.0:
    X_test_np = X_test_np / 255.0

X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)
X_val = torch.tensor(X_val_np, dtype=torch.float32)
y_val = torch.tensor(y_val_np, dtype=torch.long)
X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.long)

class_counts = np.bincount(y_train_np)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * 7
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

print(f"Train max: {X_train.max()}, Val max: {X_val.max()}")

Train max: 1.0, Val max: 1.0


In [10]:
class FinalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.5)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 7)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [11]:
BATCH_SIZE = 128
LR = 0.0005
EPOCHS = 20
WEIGHT_DECAY = 1e-3

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)

temp_model = FinalCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

temp_model.train()
start_f = time.time()
outputs = temp_model(images)
loss = criterion(outputs, labels)
f_time = time.time() - start_f

start_b = time.time()
loss.backward()
b_time = time.time() - start_b

print(f" time: Forward Pass: {f_time:.5f}s | Backward Pass: {b_time:.5f}s")

 time: Forward Pass: 0.00502s | Backward Pass: 0.00552s


In [12]:
run = wandb.init(
    project="facial-expression-recognition",
    name="Final_CNN_Inference_Run",
    config={
        "architecture": "FinalCNN",
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "epochs": EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "forward_pass_time_sec": f_time,
        "backward_pass_time_sec": b_time
    }
)

model = FinalCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print("traainiing")
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        _, preds = outs.max(1)
        correct_train += preds.eq(lbls).sum().item()
        total_train += lbls.size(0)

    train_loss /= total_train
    train_acc = (correct_train / total_train) * 100

    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outs = model(imgs)
            loss = criterion(outs, lbls)
            val_loss += loss.item() * imgs.size(0)
            _, preds = outs.max(1)
            correct_val += preds.eq(lbls).sum().item()
            total_val += lbls.size(0)

    val_loss /= total_val
    val_acc = (correct_val / total_val) * 100

    print(f"Epoch [{epoch}/{EPOCHS}] -> Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

torch.save(model.state_dict(), 'final_cnn_model.pth')
run.finish()

traainiing
Epoch [1/20] -> Train Acc: 21.98% | Val Acc: 27.14%
Epoch [2/20] -> Train Acc: 29.31% | Val Acc: 34.59%
Epoch [3/20] -> Train Acc: 35.21% | Val Acc: 41.07%
Epoch [4/20] -> Train Acc: 39.44% | Val Acc: 43.30%
Epoch [5/20] -> Train Acc: 41.46% | Val Acc: 44.28%
Epoch [6/20] -> Train Acc: 44.96% | Val Acc: 46.46%
Epoch [7/20] -> Train Acc: 46.07% | Val Acc: 45.51%
Epoch [8/20] -> Train Acc: 47.66% | Val Acc: 51.34%
Epoch [9/20] -> Train Acc: 48.65% | Val Acc: 50.43%
Epoch [10/20] -> Train Acc: 49.73% | Val Acc: 50.43%
Epoch [11/20] -> Train Acc: 50.76% | Val Acc: 53.36%
Epoch [12/20] -> Train Acc: 51.61% | Val Acc: 52.15%
Epoch [13/20] -> Train Acc: 51.67% | Val Acc: 54.35%
Epoch [14/20] -> Train Acc: 52.98% | Val Acc: 54.96%
Epoch [15/20] -> Train Acc: 53.50% | Val Acc: 53.24%
Epoch [16/20] -> Train Acc: 53.99% | Val Acc: 54.68%
Epoch [17/20] -> Train Acc: 54.70% | Val Acc: 55.65%
Epoch [18/20] -> Train Acc: 55.07% | Val Acc: 54.12%
Epoch [19/20] -> Train Acc: 55.57% | Val Acc

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_accuracy,▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇█████
train_loss,█▆▆▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁
val_accuracy,▁▃▄▅▅▅▅▇▆▆▇▇▇▇▇▇█▇▇█
val_loss,█▇▆▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁▂
epoch,20
train_accuracy,55.94214
train_loss,1.08396
val_accuracy,57.37172
val_loss,1.18019


amanac tu saocari shedegi momca testze aghar vici icit


In [13]:
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

print("its the finaal countdoownn")
model.eval()
correct_test = 0
total_test = 0

with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        outs = model(imgs)
        _, preds = outs.max(1)
        correct_test += preds.eq(lbls).sum().item()
        total_test += lbls.size(0)

test_acc = (correct_test / total_test) * 100

print("\n final results")
print(f" total tested: {total_test}")
print(f"test accuracy): {test_acc:.2f}%")

try:
    wandb.log({"final_test_accuracy": test_acc})
    torch.save(model.state_dict(), 'final_cnn_model_completed.pth')
    run.finish()
    print("all done, run finished.")
except NameError:
    print(" oops")

its the finaal countdoownn

 final results
 total tested: 3589
test accuracy): 58.82%


Error: You must call wandb.init() before wandb.log()

ui

In [14]:

run = wandb.init(
    project="facial-expression-recognition",
    name="Final_Test_Logging",
    config={"architecture": "FinalCNN"}
)

wandb.log({"final_test_accuracy": 58.82})
torch.save(model.state_dict(), 'final_cnn_model_completed.pth')
run.finish()

print(" 58.82% was logged on wandb and all is saved")

final_test_accuracy,▁
final_test_accuracy,58.82


 58.82% was logged on wandb and all is saved
